# AI Tutor on Kaggle

1. Chọn **Copy & Edit**, bật **GPU accelerator** và **Internet**.
2. Attach Kaggle Dataset/Model chứa file GGUF; attach dataset bài giảng nếu cần.
3. Tạo Kaggle Secret tên `NGROK_AUTHTOKEN`.
4. Sửa duy nhất cell **User configuration**, sau đó chọn **Run All**.
5. Mở `AI Tutor URL` được in ở cell tunnel gần cuối notebook.

In [ ]:
import shutil, subprocess

for command in (["nvidia-smi"], ["python", "--version"], ["node", "--version"], ["npm", "--version"]):
    print("$", " ".join(command))
    if shutil.which(command[0]):
        subprocess.run(command, check=False)
    else:
        print(f"WARNING: {command[0]} is not installed")
if not shutil.which("nvidia-smi"):
    print("WARNING: GPU is not enabled. Select a GPU accelerator before running the model.")

In [ ]:
# User configuration: edit values only in this cell.
REPOSITORY_URL = "https://github.com/qtrung123/AI_Tutor2-Kaggle.git"
REPOSITORY_BRANCH = "main"
PROJECT_ROOT = "/kaggle/working/AI_Tutor2"
GGUF_MODEL_PATH = "/kaggle/input/datasets/trung121212/tutor-model/qwen2.5-7b-instruct-q4_k_m.gguf"
LECTURE_INPUT_DIR = ""  # Optional read-only directory under /kaggle/input
OLLAMA_CHAT_MODEL = "qwen-tutor-7b"
OLLAMA_EMBEDDING_MODEL = "bge-m3"
BACKEND_PORT = 8000
FRONTEND_PORT = 3000
PUBLIC_PORT = 7860
TUNNEL_PROVIDER = "ngrok"
RECREATE_OLLAMA_MODEL = False

import os
for key, value in {
    "PROJECT_ROOT": PROJECT_ROOT, "GGUF_MODEL_PATH": GGUF_MODEL_PATH,
    "OLLAMA_CHAT_MODEL": OLLAMA_CHAT_MODEL,
    "OLLAMA_EMBEDDING_MODEL": OLLAMA_EMBEDDING_MODEL,
    "BACKEND_PORT": BACKEND_PORT, "FRONTEND_PORT": FRONTEND_PORT,
    "PUBLIC_PORT": PUBLIC_PORT,
    "RECREATE_OLLAMA_MODEL": int(RECREATE_OLLAMA_MODEL),
    "REBUILD_CHROMA_ON_EMBEDDING_CHANGE": 1,
}.items(): os.environ[key] = str(value)

In [ ]:
from pathlib import Path
import subprocess
root = Path(PROJECT_ROOT)
if not (root / ".git").exists():
    root.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch", REPOSITORY_URL, PROJECT_ROOT], check=True)
else:
    subprocess.run(["git", "-C", PROJECT_ROOT, "fetch", "origin", REPOSITORY_BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_ROOT, "checkout", REPOSITORY_BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_ROOT, "pull", "--ff-only", "origin", REPOSITORY_BRANCH], check=True)

In [ ]:
import shutil, subprocess
required_packages = [("nginx", "nginx"), ("curl", "curl"), ("zstd", "zstd")]
missing = [package for package, binary in required_packages if not shutil.which(binary)]
if missing:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", *missing], check=True)
if not shutil.which("ollama"):
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
subprocess.run(["ollama", "--version"], check=True)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-r", f"{PROJECT_ROOT}/requirements.txt"], check=True)

In [ ]:
import subprocess
frontend = f"{PROJECT_ROOT}/frontend"
subprocess.run(["npm", "ci"], cwd=frontend, check=True)

In [ ]:
from pathlib import Path
import shutil
data_dir = Path(PROJECT_ROOT) / "data"
data_dir.mkdir(parents=True, exist_ok=True)
(Path(PROJECT_ROOT) / "vectorstore").mkdir(parents=True, exist_ok=True)
if LECTURE_INPUT_DIR:
    source_dir = Path(LECTURE_INPUT_DIR)
    if not source_dir.is_dir(): raise FileNotFoundError(f"Lecture input not found: {source_dir}")
    for source in source_dir.rglob("*"):
        if source.is_file() and source.suffix.lower() in {".pdf", ".txt"}:
            target = data_dir / source.name
            if not target.exists() or target.stat().st_size != source.stat().st_size:
                shutil.copy2(source, target)
print("Runtime data directory:", data_dir)

In [ ]:
import subprocess
subprocess.run(["bash", f"{PROJECT_ROOT}/deployment/start_kaggle.sh"], cwd=PROJECT_ROOT, check=True)
print("Health check:")
subprocess.run(["curl", "-fsS", f"http://127.0.0.1:{PUBLIC_PORT}/api/health"], check=True)
print("\nWarming up the chat model so GPU allocation can be verified...")
import httpx
warmup = httpx.post("http://127.0.0.1:11434/api/generate", json={"model": OLLAMA_CHAT_MODEL, "prompt": "Reply with OK.", "stream": False, "keep_alive": "10m"}, timeout=600)
warmup.raise_for_status()
print("Model response:", warmup.json().get("response", "").strip())
print("Checking the embedding model with non-private test text...")
embedding_warmup = httpx.post("http://127.0.0.1:11434/api/embed", json={"model": OLLAMA_EMBEDDING_MODEL, "input": "AI Tutor embedding health check.", "keep_alive": "10m"}, timeout=120)
embedding_warmup.raise_for_status()
if not embedding_warmup.json().get("embeddings"): raise RuntimeError("Embedding warm-up returned no vector.")
print("Embedding warm-up: OK")
print("ollama list / backend health / proxied health:")
subprocess.run(["ollama", "list"], check=True)
subprocess.run(["curl", "-fsS", f"http://127.0.0.1:{BACKEND_PORT}/api/health"], check=True)
print()
subprocess.run(["curl", "-fsS", f"http://127.0.0.1:{PUBLIC_PORT}/api/health"], check=True)
print("\nLoaded models / GPU check:")
subprocess.run(["ollama", "ps"], check=False)
subprocess.run(["nvidia-smi"], check=False)

In [ ]:
import subprocess, sys
if TUNNEL_PROVIDER != "ngrok":
    print(f"Unsupported tunnel provider: {TUNNEL_PROVIDER}")
else:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyngrok"], check=True)
        from pyngrok import ngrok
        ngrok.kill()
        ngrok.set_auth_token(token)
        tunnel = ngrok.connect(addr=f"127.0.0.1:{PUBLIC_PORT}", proto="http")
        print(f"AI Tutor URL: {tunnel.public_url}")
    except Exception as error:
        print("Could not create the tunnel. Add a Kaggle Secret named NGROK_AUTHTOKEN, enable Internet, then rerun this cell.")
        print("Details:", error)

In [ ]:
import subprocess
for log_name in ["ollama.log", "backend.log", "frontend.log", "nginx.log"]:
    print(f"\n===== {log_name} =====")
    subprocess.run(["tail", "-n", "100", f"/kaggle/working/ai-tutor-logs/{log_name}"], check=False)
subprocess.run(["ollama", "list"], check=False)
subprocess.run(["ollama", "ps"], check=False)
subprocess.run(["nvidia-smi"], check=False)
ps = subprocess.run(["ollama", "ps"], capture_output=True, text=True, check=False).stdout
if ps.strip() and "GPU" not in ps.upper(): print("WARNING: Ollama does not report GPU usage; inspect PROCESSOR and nvidia-smi.")